### identify the env

two dataset strategy:

a. data from all nodes -> into a dataset and split to train 0.75/test 0.25

b. data from 3 node (0, 1, 3) -> for train / 1 node -> test (2)


500 seq length as frame

model: ResNet

overlapping: 40%

report:

model.summary() 

hyper parameter (overlapping / learning rate = 0.001)

In [113]:
import pandas as pd
import numpy as np
import glob

WINDOW_SIZE = 500 # 100/500/1000
STRIDE = 40 # 40/50

env_files = [
    "env/e0-bridge.csv",
    "env/e1-lake.csv",
    "env/e2-forest.csv",
    "env/e3-river.csv",
    "env/e4-garden.csv"
]

device_to_label = {
    "RIOT-BLE-0": 0,
    "RIOT-BLE-1": 1,
    "RIOT-BLE-2": 2,
    "RIOT-BLE-3": 3
}

node_files = [
    "node/node0.csv",
    "node/node1.csv",
    "node/node2.csv",
    "node/node3.csv"
]

env_to_label = {
    "env/e0-bridge.csv": 0,
    "env/e1-lake.csv": 1,
    "env/e2-forest.csv": 2,
    "env/e3-river.csv": 3,
    "env/e4-garden.csv": 4
}

In [114]:
# === create node csvs ===
from pathlib import Path

all_df = []

for file in env_files:
    df = pd.read_csv(file)  

    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df = df.dropna(subset=["ts"]).copy()
    df["source_file"] = file

    all_df.append(df) # collect all dataframes for later use

df_all = pd.concat(all_df, ignore_index=True) # combine all dataframes into one

Path("node").mkdir(exist_ok=True)
for device, node_name in device_to_label.items():
    sub = df_all[df_all["device"] == device].copy() # filter by device
    sub = sub.sort_values("ts")
    sub.to_csv(f"node/node{node_name}.csv", index=False) # save to node csv
    
    print(f"node{node_name}.csv:  {len(sub)} rows")

node0.csv:  98602 rows
node1.csv:  82109 rows
node2.csv:  77771 rows
node3.csv:  82140 rows


In [115]:
# === store data ===
X = []
y = []
node_ids = []

# process each node file
for node_id, file in enumerate(node_files):
    df = pd.read_csv(file)

    df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
    df = df.dropna(subset=["ts"]).sort_values("ts")

    for env_name, env_label in env_to_label.items():
        df_env = df[df["source_file"] == env_name].copy()

        if len(df_env) < WINDOW_SIZE:
            continue

        df_env["rssi_diff"] = df_env["rssi"].diff()

        df_env = df_env.dropna(subset=["rssi_diff"])

        y_min = df_env["rssi_diff"].min()
        y_max = df_env["rssi_diff"].max()

        if y_max - y_min == 0:
            continue

        df_env["rssi_norm"] = (df_env["rssi_diff"] - y_min) / (y_max - y_min)

        data = df_env["rssi_norm"].values

        for i in range(0, len(data) - WINDOW_SIZE, STRIDE):
            seq = data[i:i+WINDOW_SIZE]
            X.append(seq)
            y.append(env_label)
            node_ids.append(node_id)

X = np.array(X)
y = np.array(y)
node_ids = np.array(node_ids)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("node_ids shape:", node_ids.shape)

X = X.astype(np.float32)
y = y.astype(np.int64)
node_ids = node_ids.astype(np.int64)

# PyTorch Conv1d input: (batch, channels, length)
X_data = X[:, np.newaxis, :]  
print("X_data shape:", X_data.shape)

X shape: (8277, 500)
y shape: (8277,)
node_ids shape: (8277,)
X_data shape: (8277, 1, 500)


In [116]:
import numpy as np
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import TensorDataset, DataLoader

# =========================
# Version 1: random split
# =========================
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_data, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# turn into PyTorch tensors
X_train_r_tensor = torch.tensor(X_train_r, dtype=torch.float32)
X_test_r_tensor  = torch.tensor(X_test_r, dtype=torch.float32)
y_train_r_tensor = torch.tensor(y_train_r, dtype=torch.long)
y_test_r_tensor  = torch.tensor(y_test_r, dtype=torch.long)

# create Dataset and DataLoader
train_dataset_r = TensorDataset(X_train_r_tensor, y_train_r_tensor)
test_dataset_r  = TensorDataset(X_test_r_tensor, y_test_r_tensor)

train_loader_r = DataLoader(train_dataset_r, batch_size=64, shuffle=True)
test_loader_r  = DataLoader(test_dataset_r, batch_size=64, shuffle=False)

print("\n=== Version 1: Random Split ===")
print("X_train:", X_train_r_tensor.shape)
print("X_test :", X_test_r_tensor.shape)
print("y_train:", y_train_r_tensor.shape)
print("y_test :", y_test_r_tensor.shape)


=== Version 1: Random Split ===
X_train: torch.Size([6207, 1, 500])
X_test : torch.Size([2070, 1, 500])
y_train: torch.Size([6207])
y_test : torch.Size([2070])


In [117]:
# =========================
# Version 2: node-based split
# =========================
train_nodes = [0, 1, 3]   
test_node = 2             

train_mask = np.isin(node_ids, train_nodes)
test_mask = (node_ids == test_node)

X_train_n = X_data[train_mask]
y_train_n = y[train_mask]

X_test_n = X_data[test_mask]
y_test_n = y[test_mask]

# turn into PyTorch tensors
X_train_n_tensor = torch.tensor(X_train_n, dtype=torch.float32)
X_test_n_tensor  = torch.tensor(X_test_n, dtype=torch.float32)
y_train_n_tensor = torch.tensor(y_train_n, dtype=torch.long)
y_test_n_tensor  = torch.tensor(y_test_n, dtype=torch.long)

print("X_train_n_tensor shape:", X_train_n_tensor.shape)
print("y_train_n_tensor shape:", y_train_n_tensor.shape)

# create Dataset and DataLoader
train_dataset_n = TensorDataset(X_train_n_tensor, y_train_n_tensor)
test_dataset_n  = TensorDataset(X_test_n_tensor, y_test_n_tensor)

train_loader_n = DataLoader(train_dataset_n, batch_size=64, shuffle=True)
test_loader_n  = DataLoader(test_dataset_n, batch_size=64, shuffle=False)

print("\n=== Version 2: Env-based Split ===")
print("Train nodes:", train_nodes)
print("Test node :", test_node)
print("X_train:", X_train_n_tensor.shape)
print("X_test :", X_test_n_tensor.shape)
print("y_train:", y_train_n_tensor.shape)
print("y_test :", y_test_n_tensor.shape)

X_train_n_tensor shape: torch.Size([6393, 1, 500])
y_train_n_tensor shape: torch.Size([6393])

=== Version 2: Env-based Split ===
Train nodes: [0, 1, 3]
Test node : 2
X_train: torch.Size([6393, 1, 500])
X_test : torch.Size([1884, 1, 500])
y_train: torch.Size([6393])
y_test : torch.Size([1884])


In [118]:
# === Residual Block ===
import torch.nn as nn

class ResidualBlock1D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv1d(
            in_channels, out_channels,
            kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm1d(out_channels) # batch norm after conv1
        self.relu = nn.ReLU(inplace=True)

        # self.dropout = nn.Dropout(p=0.1) # dropout layer

        self.conv2 = nn.Conv1d(
            out_channels, out_channels,
            kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm1d(out_channels) # batch norm after conv2

        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels: # projection shortcut if dimensions differ
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_channels)
            )
    
    # forward pass
    def forward(self, x):
        identity = self.shortcut(x)

        # conv1 -> bn -> relu
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        # out = self.dropout(out) # dropout 

        # conv2 -> bn
        out = self.conv2(out)
        out = self.bn2(out)

        # add shortcut
        out += identity
        out = self.relu(out)

        return out

In [119]:
# === ResNet1D Model ===

class ResNet1D(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()

        # initial convolution and pooling
        self.stem = nn.Sequential(
            nn.Conv1d(1, 16, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        )

        # layer1
        self.layer1 = nn.Sequential(
            ResidualBlock1D(16, 16, stride=1)
        )

        # layer2
        self.layer2 = nn.Sequential(
            ResidualBlock1D(16, 32, stride=2)
        )

        # layer3
        self.layer3 = nn.Sequential(
            ResidualBlock1D(32, 64, stride=2)
        )

        self.global_pool = nn.AdaptiveAvgPool1d(1)
        # self.dropout = nn.Dropout(p=0.3)
        self.fc = nn.Linear(64, num_classes)

    def forward(self, x):
        x = self.stem(x)         # (B, 1, 100) -> (B, 16, 25)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.global_pool(x)  # (B, 64, 1)
        x = x.squeeze(-1)        # (B, 64, 1) -> (B, 64)
        # x = self.dropout(x)      # dropout
        x = self.fc(x)           # (B, num_classes)
        return x

num_classes = len(np.unique(y))
model = ResNet1D(num_classes=num_classes)
print(model)

ResNet1D(
  (stem): Sequential(
    (0): Conv1d(1, 16, kernel_size=(7,), stride=(2,), padding=(3,), bias=False)
    (1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool1d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  )
  (layer1): Sequential(
    (0): ResidualBlock1D(
      (conv1): Conv1d(16, 16, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
      (bn1): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv1d(16, 16, kernel_size=(3,), stride=(1,), padding=(1,), bias=False)
      (bn2): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
  )
  (layer2): Sequential(
    (0): ResidualBlock1D(
      (conv1): Conv1d(16, 32, kernel_size=(3,), stride=(2,), padding=(1,), bias=False)
      (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=T

In [120]:
# =========================
# Step 3: training setup
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [121]:
# =========================
# Step 4: training loop
# =========================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        outputs = model(xb)
        loss = criterion(outputs, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    return total_loss / total, correct / total


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)

            outputs = model(xb)
            loss = criterion(outputs, yb)

            total_loss += loss.item() * xb.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    return total_loss / total, correct / total

In [122]:
# =========================
# Step 5: run training
# =========================
num_epochs = 20


def train_and_evaluate(model, train_loader, test_loader, criterion, optimizer, device, num_epochs):
      best_test_loss = float("inf")
      best_state = None
      for epoch in range(num_epochs):
            train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
            test_loss, test_acc = evaluate(model, test_loader, criterion, device)

            print(f"Epoch [{epoch+1}/{num_epochs}] "
                  f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} | "
                  f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.4f}")

      # save best model
      if test_loss < best_test_loss:
          best_test_loss = test_loss
          best_state = model.state_dict()
      model.load_state_dict(best_state)

# ===========================
# version 1: random split
# ===========================

print("\n=== Version 1: random split ===\n")
train_and_evaluate(model, train_loader_r, test_loader_r, criterion, optimizer, device, num_epochs)


=== Version 1: random split ===

Epoch [1/20] Train Loss: 0.9663, Train Acc: 0.5958 | Test Loss: 0.7989, Test Acc: 0.7077
Epoch [2/20] Train Loss: 0.6577, Train Acc: 0.7226 | Test Loss: 0.5655, Test Acc: 0.7816
Epoch [3/20] Train Loss: 0.5083, Train Acc: 0.7967 | Test Loss: 0.5127, Test Acc: 0.7826
Epoch [4/20] Train Loss: 0.4454, Train Acc: 0.8254 | Test Loss: 0.3907, Test Acc: 0.8478
Epoch [5/20] Train Loss: 0.4304, Train Acc: 0.8342 | Test Loss: 0.3833, Test Acc: 0.8488
Epoch [6/20] Train Loss: 0.3525, Train Acc: 0.8764 | Test Loss: 0.3840, Test Acc: 0.8425
Epoch [7/20] Train Loss: 0.2502, Train Acc: 0.9191 | Test Loss: 0.5504, Test Acc: 0.7778
Epoch [8/20] Train Loss: 0.2111, Train Acc: 0.9333 | Test Loss: 0.1308, Test Acc: 0.9647
Epoch [9/20] Train Loss: 0.1907, Train Acc: 0.9378 | Test Loss: 0.2195, Test Acc: 0.9130
Epoch [10/20] Train Loss: 0.1565, Train Acc: 0.9518 | Test Loss: 0.1711, Test Acc: 0.9333
Epoch [11/20] Train Loss: 0.1142, Train Acc: 0.9678 | Test Loss: 0.2942, Te

In [123]:
# =========================
# Step 6: prediction example
# =========================

def predict(model, X_test_tensor, y_test, device):
    model.eval()

    with torch.no_grad():
        sample_x = X_test_tensor[:5].to(device)
        outputs = model(sample_x)
        preds = outputs.argmax(dim=1).cpu().numpy()

    print("Pred:", preds)
    print("True:", y_test[:5])

predict(model, X_test_r_tensor, y_test_r, device)

Pred: [2 3 1 0 3]
True: [2 3 1 0 3]


In [124]:
# =========================
# Step 7: confusion matrix
# =========================
from sklearn.metrics import confusion_matrix, classification_report

def compute_confusion_matrix(model, loader, device):
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            outputs = model(xb)
            preds = outputs.argmax(dim=1).cpu().numpy()

            all_preds.extend(preds)
            all_true.extend(yb.numpy())

    cm = confusion_matrix(all_true, all_preds)
    print("Confusion Matrix:\n", cm)
    print("\nClassification Report:\n", classification_report(all_true, all_preds))

print("=== Version 1: random split ===\n")
compute_confusion_matrix(model, test_loader_r, device)

=== Version 1: random split ===

Confusion Matrix:
 [[487   0   0   0   0]
 [  0 270   0   0   0]
 [  0   0 466   4   0]
 [  0   0   0 459   0]
 [  0   0   0   0 384]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       487
           1       1.00      1.00      1.00       270
           2       1.00      0.99      1.00       470
           3       0.99      1.00      1.00       459
           4       1.00      1.00      1.00       384

    accuracy                           1.00      2070
   macro avg       1.00      1.00      1.00      2070
weighted avg       1.00      1.00      1.00      2070



In [125]:
# ===========================
# version 2: node-based split
# ===========================

print("\n=== Version 2: node-based split ===\n")
train_and_evaluate(model, train_loader_n, test_loader_n, criterion, optimizer, device, num_epochs)


=== Version 2: node-based split ===

Epoch [1/20] Train Loss: 0.0401, Train Acc: 0.9880 | Test Loss: 0.0795, Test Acc: 0.9687
Epoch [2/20] Train Loss: 0.0161, Train Acc: 0.9981 | Test Loss: 0.1935, Test Acc: 0.9183
Epoch [3/20] Train Loss: 0.0212, Train Acc: 0.9952 | Test Loss: 0.1344, Test Acc: 0.9485
Epoch [4/20] Train Loss: 0.0243, Train Acc: 0.9933 | Test Loss: 0.2703, Test Acc: 0.8907
Epoch [5/20] Train Loss: 0.0133, Train Acc: 0.9970 | Test Loss: 0.1552, Test Acc: 0.9358
Epoch [6/20] Train Loss: 0.0328, Train Acc: 0.9919 | Test Loss: 1.2180, Test Acc: 0.7548
Epoch [7/20] Train Loss: 0.0196, Train Acc: 0.9945 | Test Loss: 0.2017, Test Acc: 0.9071
Epoch [8/20] Train Loss: 0.0303, Train Acc: 0.9897 | Test Loss: 0.1927, Test Acc: 0.9299
Epoch [9/20] Train Loss: 0.0253, Train Acc: 0.9926 | Test Loss: 1.4616, Test Acc: 0.6545
Epoch [10/20] Train Loss: 0.0252, Train Acc: 0.9928 | Test Loss: 0.4784, Test Acc: 0.8445
Epoch [11/20] Train Loss: 0.0108, Train Acc: 0.9973 | Test Loss: 1.0017

In [126]:
predict(model, X_test_n_tensor, y_test_n, device)

Pred: [2 2 3 0 0]
True: [0 0 0 0 0]


In [127]:
print("=== Version 2: node-based split ===\n")
compute_confusion_matrix(model, test_loader_n, device)

=== Version 2: node-based split ===

Confusion Matrix:
 [[203   0   3  73   0]
 [  0 182   0   3   0]
 [  0   0 571  17   8]
 [  0  10  43 383  30]
 [  0   0   0   0 358]]

Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.73      0.84       279
           1       0.95      0.98      0.97       185
           2       0.93      0.96      0.94       596
           3       0.80      0.82      0.81       466
           4       0.90      1.00      0.95       358

    accuracy                           0.90      1884
   macro avg       0.92      0.90      0.90      1884
weighted avg       0.90      0.90      0.90      1884

